# GraphAgents: Knowledge Graph-Guided Agentic AI for Cross-Domain Materials Design

#### Authors: Isabella Stewart, Tarjei Hage, Yu-Chuan (Michael) Hsu, and Markus J. Buehler, MIT, 2025 
#### Corresponding author: Markus J. Buehler, mbuehler@MIT.EDU
#### LAMM, Massachusetts Institute of Technology

In [2]:
import os
from GraphReasoning import *


In [3]:
verbatim=False

In [4]:
doc_data_dir = '/home/mkychsu/pool/SG_materialproperties/'
data_dir='./GRAPHDATA'    
data_dir_output='./GRAPHDATA_OUTPUT'

filename = 'Meta-Llama-3.3-70B-Instruct-Q4_K_L.gguf'
n_ctx = 20000

embedding_file='SG_LLAMA33_70b.pkl'
model_id = "/home/mkychsu/pool/llm/Cephalo-Phi-3-vision-128k-4b-alpha"


### Load dataset

In [5]:
import pandas as pd
import glob
try:
    df = pd.read_csv(f'{doc_data_dir}/materialproperties.csv', index_col=0)
    
except:
    
    doc_list=sorted(glob.glob(f'{doc_data_dir}/*.xls'))
    df_list = []
    for i, doc in enumerate(doc_list):
        print(i, doc)
        df_list.append(pd.read_excel(doc))
        
    df = pd.concat(df_list, axis=0)
    df = df.drop_duplicates()
    df = df.reset_index(drop=True)
    df.to_csv(f'{doc_data_dir}/materialproperties.csv', drop_index=True)


0 /home/mkychsu/pool/SG_abstracts/Biocompatible and Non-adherence.xls
1 /home/mkychsu/pool/SG_abstracts/Biocompatible and Non-reactive.xls
2 /home/mkychsu/pool/SG_abstracts/Broad Temperature Stability.xls
3 /home/mkychsu/pool/SG_abstracts/Chemical_Resistance_to_Methanol.xls
4 /home/mkychsu/pool/SG_abstracts/Chemical_Resistance_to_Protein_Denaturants.xls
5 /home/mkychsu/pool/SG_abstracts/Chemical_Resistance_to_acetone.xls
6 /home/mkychsu/pool/SG_abstracts/Chemical_resistance_to_DMSO.xls
7 /home/mkychsu/pool/SG_abstracts/Chemical_resistance_to_guanine_hydrochloride.xls
8 /home/mkychsu/pool/SG_abstracts/Chemical_resistance_to_organic_solvents.xls
9 /home/mkychsu/pool/SG_abstracts/Chemical_resistance_to_strong_acid.xls
10 /home/mkychsu/pool/SG_abstracts/Compatible with Sterilization.xls
11 /home/mkychsu/pool/SG_abstracts/Extreme_Thermal_resistance.xls
12 /home/mkychsu/pool/SG_abstracts/Extreme_chemical_resistance.xls
13 /home/mkychsu/pool/SG_abstracts/Flexible at Low Temperatures.xls
14 /h

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer_model=f'/home/mkychsu/pool/llm/nomic-embed-text-v1.5'

from sentence_transformers import SentenceTransformer

embedding_tokenizer =''
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)


from GraphReasoning import load_embeddings, save_embeddings, generate_node_embeddings
from transformers import AutoProcessor 

model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda:1", trust_remote_code=True, torch_dtype="auto")
processor = AutoProcessor.from_pretrained(model_id, device_map="cuda:1", trust_remote_code=True) 



/home/mkychsu/pool/.conda/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mkychsu/pool/.conda/envs/llm/lib/python3.12/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:410: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/mkychsu/pool/.conda/envs/llm/lib/python3.12/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:418: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/home/mkychsu/pool/.conda/envs/llm/lib/python3.12/site-packages/auto_gptq/nn_modules/triton_utils/kernels.py:461: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torc

ENV: Auto setting PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True' for memory saving.
ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for compatibililty.


`low_cpu_mem_usage` was None, now default to True since model is quantized.
/home/mkychsu/pool/.conda/envs/llm/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:818: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(
INFO - Auto pick kernel based on compatibility: <class 'gptqmodel.nn_modules.qlinear.torch.TorchQuantLinear'>
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.94it/s]
INFO:transformers_modules.microsoft.Phi-3-vision-128k-instruct.c45209e90a4c4f7d16b2e9d48503c7f3e83623ed.image_embedding_phi3_v:learnable separator enabled for hd transform, hd_transform_order = sub_glb
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.27s/it]
/home/mkychsu/pool/.conda/envs/llm/lib/python3.12/site-

In [7]:
import torch

if os.path.exists(f'{data_dir}/{embedding_file}'):
    generate_new_embeddings=False

generate_new_embeddings=True

with torch.no_grad():
    if generate_new_embeddings:

        try:
            import networkx as nx

            graph_root=embedding_file.split('.')[0]
            graph_GraphML= f'{data_dir_output}/{graph_root}.graphml'
            G = nx.read_graphml(graph_GraphML)
            node_embeddings = generate_node_embeddings(G, embedding_tokenizer, embedding_model, )
        except:
            node_embeddings = generate_node_embeddings(nx.DiGraph(), embedding_tokenizer, embedding_model, )

        save_embeddings(node_embeddings, f'{data_dir}/{embedding_file}')

    else:
        filename = f"{data_dir}/{embedding_file}"
        node_embeddings = load_embeddings(f'{data_dir}/{embedding_file}')
        
        


0it [00:00, ?it/s]


### Set up LLM client:

In [8]:
from llama_cpp import Llama

from llama_cpp.llama_speculative import LlamaPromptLookupDecoding

llm = Llama(model_path=file_path,
             n_gpu_layers=-1,verbose= True, #False,#False,
             n_ctx=n_ctx,
             main_gpu=0,
             n_threads= 4,
             n_threads_batch=32,
             draft_model=LlamaPromptLookupDecoding(num_pred_tokens=2),
             logits_all=True,
             # chat_format='mistral-instruct',
             )
# In[10]:

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 6 CUDA devices:
  Device 0: Tesla V100S-PCIE-32GB, compute capability 7.0, VMM: yes
  Device 1: Tesla V100S-PCIE-32GB, compute capability 7.0, VMM: yes
  Device 2: Tesla V100S-PCIE-32GB, compute capability 7.0, VMM: yes
  Device 3: Tesla V100S-PCIE-32GB, compute capability 7.0, VMM: yes
  Device 4: Tesla V100S-PCIE-32GB, compute capability 7.0, VMM: yes
  Device 5: Tesla V100S-PCIE-32GB, compute capability 7.0, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (Tesla V100S-PCIE-32GB) - 26708 MiB free
llama_model_load_from_file_impl: using device CUDA1 (Tesla V100S-PCIE-32GB) - 24262 MiB free
llama_model_load_from_file_impl: using device CUDA2 (Tesla V100S-PCIE-32GB) - 32184 MiB free
llama_model_load_from_file_impl: using device CUDA3 (Tesla V100S-PCIE-32GB) - 32184 MiB free
llama_model_load_from_file_impl: using device CUDA4 (Tesla V100S-PCIE-32GB) - 32184 MiB free
lla

In [9]:

import instructor
from typing import List
from PIL import Image

from pydantic import BaseModel

class Node(BaseModel):
    id: str
    type: str
        
class Edge(BaseModel):
    source: str
    target: str
    relation: str
        
class KnowledgeGraph(BaseModel):
    nodes: List[Node]
    edges: List[Edge]

response_model = KnowledgeGraph
system_prompt = '''
You are a scientific assistant extracting knowledge graphs from text.
Return a JSON with two fields: <nodes> and <edges>.\n
Each node must have <id> and <type>.\n
Each edge must have <source>, <target>, and <relation>.
'''

def generate(system_prompt=system_prompt, 
             prompt="",temperature=0.333,
             max_tokens=n_ctx, response_model=KnowledgeGraph, 
            ):     

    if system_prompt==None:
        messages=[
            {"role": "user", "content": f"{prompt}"},
        ]

    else:
        messages=[
            {"role": "system",  "content": f"{system_prompt}"},
            {"role": "user", "content": f"{prompt}"},
        ]

    if 'json' in prompt.lower() and 'graph' in prompt.lower():
        create = instructor.patch(
            create=llm.create_chat_completion_openai_v1,
            mode=instructor.Mode.JSON_SCHEMA,
        )

        result = create(messages=messages, 
                        temperature=temperature,
                        max_tokens=max_tokens,
                        response_model=response_model,
                       )
        return result
    else:
        
        result=llm.create_chat_completion_openai_v1(
    
        # result=llm.create_chat_completion(
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
        return result.choices[0].message.content #['choices'][0]['message']['content']



def generate_figure(image, system_prompt=system_prompt, 
                prompt="", model=model, processor=processor, temperature=0,
                           ):
    if system_prompt==None:
        messages=[
            {"role": "user", "content": f"Here is the image: <|image_1|>.\n" + prompt},
        ]

    else:
        messages=[
            {"role": "system",  "content": system_prompt},
            {"role": "user", "content": f"Here is the image: <|image_1|>.\n" + prompt},
        ]
        
    try:
        pwd = os.getcwd()
        image = image.split(pwd)[-1]
        image=Path('.').glob(f'**/{image}', case_sensitive=False)
        image = list(image)[0]
    except:
        return '' 
    image = Image.open(image)
    print(f'Extracting infomation from {image}')
    prompt = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(prompt, [image], return_tensors="pt").to("cuda:1") 
    generation_args = { 
                        "max_new_tokens": 1024, 
                        "temperature": 0.1, 
                        "do_sample": True, 
                        "stop_strings": ['<|end|>',
                                         '<|endoftext|>'],
                        "tokenizer": processor.tokenizer,
                      } 

    generate_ids = model.generate(**inputs, eos_token_id=processor.tokenizer.eos_token_id, **generation_args) 

    # remove input tokens 
    generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
    return processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0] 
    

In [ ]:
import networkx as nx
from GraphReasoning import make_graph_from_text, add_new_subgraph_from_text, save_embeddings
G=nx.DiGraph()
with torch.no_grad():
    # for i, doc in enumerate(doc_list):
    for i, row in df.iterrows():

        title = row['Article Title']
        title=title.replace('/','|')
        doi = row['DOI']
        graph_root = f'{title}'

        _graph_GraphML= f'{data_dir_output}/{graph_root}_augmented_graphML_integrated.graphml'
        txt=row['Abstract']
        # print(f'{doc}')
        image_list = ''#glob.glob('/'.join(doc.split('/')[:-1])+'/*png')
        # break

        current_graph = f'{data_dir}/{graph_root}_graph.graphml'
        if os.path.exists(_graph_GraphML):
            G = nx.read_graphml(_graph_GraphML)
            print(f'Main KG loaded: {_graph_GraphML}, {G}')
            continue
            
        if os.path.exists(f'{title}_err.txt'):
            print(f'No. {i}: {title} got something wrong.')
            continue

        if not os.path.exists(current_graph):
            _, current_graph, _, _, _ = make_graph_from_text(txt,generate,
                                  generate_figure, image_list,
                                  graph_root=graph_root,
                                  chunk_size=2000,chunk_overlap=500,
                                  repeat_refine=0,verbatim=False,
                                  data_dir=data_dir,
                                  save_PDF=False,#TO DO
                                 )
 
        
        print(f'Merging graph No. {i}: {title} to the main one')
        _, G, _, node_embeddings, _ = add_new_subgraph_from_text(txt='',
                           node_embeddings=node_embeddings,
                           tokenizer=embedding_tokenizer,
                           model=embedding_model,
                           original_graph=G, data_dir_output=data_dir_output, graph_root=graph_root,
                           do_simplify_graph=True,size_threshold=10,
                           repeat_refine=0,similarity_threshold=0.97,
                           do_Louvain_on_new_graph=True,
                           #whether or not to simplify, uses similiraty_threshold defined above
                           return_only_giant_component=False,
                           save_common_graph=False,G_to_add=None,graph_GraphML_to_add=current_graph,
                           verbatim=True,)
        save_embeddings(node_embeddings, f'{data_dir}/{embedding_file}')

       

### Test

In [20]:
print(title)

Broad-temperature-span and improved piezoelectric/dielectric properties in potassium sodium niobate-based ceramics through diffusion phase transition
